# Week 11 — Tool Calling & Chatbot
### Phase 4: Generative AI

This week we move from *single prompts* to *conversational systems*. A chatbot is more than an LLM call: it has to **act** (call tools), **remember** (manage memory), and **keep track of who it's talking to and what's going on** (state).

| Part | Topic | What you'll build |
|---|---|---|
| 1 | Function / tool calling | A tool-execution loop from scratch |
| 2 | Multi-turn conversations | A stateful chat session + context-window trimming |
| 3 | Memory — buffer, summary, vector | Five memory strategies behind one interface |
| 4 | Conversation state management | Sessions, structured user state, persistence |
| 🎯 | **Project** | **A single-class chatbot with full memory handling (LangChain)** |

**Learning objectives** — by the end you should be able to:
1. Explain what really happens during a tool call (the LLM never runs your code!).
2. Write a robust agent loop that handles multiple, parallel and failing tool calls.
3. Explain why LLMs are stateless and how chat history fakes "memory".
4. Choose between buffer, window, token, summary and vector memory for a given use case.
5. Manage multiple sessions, extract structured state, and persist conversations to disk.

## 0. Setup

We use **LangChain** (`langchain-core` + `init_chat_model`) so the same code works with any provider. Pick one below and install its integration package.

| Provider | Package | Needs | Notes |
|---|---|---|---|
| `openai` | `langchain-openai` | `OPENAI_API_KEY` | |
| `anthropic` | `langchain-anthropic` | `ANTHROPIC_API_KEY` | |
| `groq` | `langchain-groq` | `GROQ_API_KEY` | Free tier available |
| `ollama` | `langchain-ollama` | Local Ollama running | Fully offline; use a tool-capable model e.g. `llama3.1` / `qwen2.5` |

For **vector memory** we use a free local embedding model via `langchain-huggingface` (no API key).

In [ ]:
# Install once (uncomment the provider you use)
%pip install -q -U langchain langchain-core pydantic langchain-huggingface sentence-transformers
%pip install -q -U langchain-openai
# %pip install -q -U langchain-anthropic
# %pip install -q -U langchain-groq
# %pip install -q -U langchain-ollama

In [ ]:
import os, getpass

PROVIDER = "openai" 

DEFAULT_MODELS = {"openai":    "gpt-4o-mini"}
KEY_NAMES = {"openai": "OPENAI_API_KEY"}

MODEL = DEFAULT_MODELS[PROVIDER]

if PROVIDER in KEY_NAMES and not os.environ.get(KEY_NAMES[PROVIDER]):
    os.environ[KEY_NAMES[PROVIDER]] = getpass.getpass(f"Enter {KEY_NAMES[PROVIDER]}: ")

In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL, model_provider=PROVIDER, temperature=0)

# Sanity check
print(llm.invoke("Say 'ready' and nothing else.").content)

Ready.


In [7]:
# Small helpers used throughout the notebook
from langchain_core.messages import BaseMessage

def text_of(msg: BaseMessage) -> str:
    # Return the plain text of a message. Some providers (e.g. Anthropic) return a list of
    # content blocks instead of a string, especially when tools are involved.
    c = msg.content
    if isinstance(c, str):
        return c
    return "".join(b.get("text", "") for b in c if isinstance(b, dict) and b.get("type") == "text")

def show(messages):
    # Pretty-print a list of messages.
    for m in messages:
        m.pretty_print()

---
# Part 1 — Function / Tool Calling

## 1.1 The core idea

An LLM can only produce text. **Tool calling** is a protocol where the model produces *structured text* that says "please call function `X` with arguments `Y`". **Your code** runs the function and sends the result back.

```
 ┌──────────┐  1. question + tool schemas   ┌─────────┐
 │          │ ────────────────────────────► │         │
 │   Your   │  2. AIMessage.tool_calls      │   LLM   │
 │   code   │ ◄──────────────────────────── │         │
 │          │                               └─────────┘
 │  3. run the Python function locally
 │          │  4. ToolMessage(result)       ┌─────────┐
 │          │ ────────────────────────────► │   LLM   │
 │          │  5. final natural-language    │         │
 │          │ ◄──────────────────────────── └─────────┘
 └──────────┘
```

> 🔑 **The model never executes anything.** It only *chooses* a tool and *fills in arguments*, based on the tool's **name, description and parameter schema**. Writing good descriptions is prompt engineering (Week 10) applied to functions.

## 1.2 Defining tools

The `@tool` decorator turns a normal Python function into a tool. LangChain reads:
- the **function name** → tool name
- the **docstring** → description the model sees
- the **type hints** → JSON schema for arguments

In [ ]:
import ast, operator, math
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_core.tools import tool

# ---- Tool 1: a *safe* calculator (never use eval() on model output!) ----
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}
_FUNCS = {"sqrt": math.sqrt, "log": math.log, "sin": math.sin, "cos": math.cos, "abs": abs, "round": round}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _FUNCS:
        return _FUNCS[node.func.id](*[_safe_eval(a) for a in node.args])
    raise ValueError("Unsupported expression")

@tool
def calculator(expression: str) -> str:
    '''Evaluate a math expression, e.g. "23 * (4 + 5)" or "sqrt(2) ** 3".
    Supports + - * / ** %, and sqrt, log, sin, cos, abs, round.'''
    return str(_safe_eval(ast.parse(expression, mode="eval")))

# ---- Tool 2: current time in a timezone ----
@tool
def get_current_time(timezone: str = "UTC") -> str:
    '''Get the current date and time in an IANA timezone such as "Asia/Kolkata",
    "Europe/London" or "America/New_York".'''
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")

# ---- Tool 3: a mock weather API (replace with a real API later) ----
_FAKE_WEATHER = {
    "mumbai": {"temp_c": 31, "condition": "humid, light rain"},
    "delhi": {"temp_c": 34, "condition": "hazy sunshine"},
    "london": {"temp_c": 14, "condition": "overcast"},
    "tokyo": {"temp_c": 22, "condition": "clear"},
}

@tool
def get_weather(city: str, date: datetime) -> dict:
    '''Get the current weather for a city. Returns temperature in Celsius and a condition.'''
    data = _FAKE_WEATHER.get(city.strip().lower())
    if data is None:
        raise ValueError(f"No weather data for '{city}'. Known: {list(_FAKE_WEATHER)}")
    return {"city": city, **data}

tools = [calculator, get_current_time, get_weather]
tools_by_name = {t.name: t for t in tools}

# Tools are Runnables — you can call them directly
print(calculator.invoke({"expression": "23 * (4 + 5)"}))
print(get_weather.invoke({"city": "Tokyo"}))

207
{'city': 'Tokyo', 'temp_c': 22, 'condition': 'clear'}


### What the model actually sees
Every provider receives something like the JSON schema below. Look at how the docstring and type hints were converted.

In [4]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

print(json.dumps(convert_to_openai_tool(get_current_time), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_current_time",
    "description": "Get the current date and time in an IANA timezone such as \"Asia/Kolkata\",\n    \"Europe/London\" or \"America/New_York\".",
    "parameters": {
      "properties": {
        "timezone": {
          "default": "UTC",
          "type": "string"
        }
      },
      "type": "object"
    }
  }
}


## 1.3 Binding tools and inspecting the model's decision

`llm.bind_tools(tools)` attaches the schemas to every request. The response is an `AIMessage`; if the model wants a tool, `.tool_calls` is a non-empty list of `{"name", "args", "id"}`.

In [8]:
llm_with_tools = llm.bind_tools(tools)

ai_msg = llm_with_tools.invoke("What's the weather like in London right now?")
print("Text content :", repr(text_of(ai_msg)))
print("Tool calls   :", ai_msg.tool_calls)

Text content : ''
Tool calls   : [{'name': 'get_weather', 'args': {'city': 'London'}, 'id': 'call_BSXfJpdM1dD8xSOvcmmphlEX', 'type': 'tool_call'}]


In [9]:
# A question that needs NO tool — the model should just answer
ai_msg = llm_with_tools.invoke("What is the capital of Japan?")
print("Text content :", text_of(ai_msg))
print("Tool calls   :", ai_msg.tool_calls)

Text content : The capital of Japan is Tokyo.
Tool calls   : []


## 1.4 Closing the loop: executing tools and returning results

The result of each tool call goes back as a `ToolMessage` whose `tool_call_id` matches the call's `id`. The **order matters**:

```
HumanMessage → AIMessage(tool_calls=[...]) → ToolMessage(s) → AIMessage(final answer)
```

In [10]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

messages = [HumanMessage("What's the weather in Mumbai, and what's 31 degrees Celsius in Fahrenheit?")]

# Step 1: model decides
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: we execute every requested tool
for tc in ai_msg.tool_calls:
    result = tools_by_name[tc["name"]].invoke(tc["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"], name=tc["name"]))

# Step 3: model reads the results and answers (it may request more tools — handled in 1.5)
final = llm_with_tools.invoke(messages)
messages.append(final)

show(messages)

================================ Human Message =================================

What's the weather in Mumbai, and what's 31 degrees Celsius in Fahrenheit?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_ZBGoZUE8zAamNVgVjAlxZVlI)
 Call ID: call_ZBGoZUE8zAamNVgVjAlxZVlI
  Args:
    city: Mumbai
  calculator (call_jfSfnFxjHIpo0tUUoMkpLiwp)
 Call ID: call_jfSfnFxjHIpo0tUUoMkpLiwp
  Args:
    expression: (31 * 9/5) + 32
================================= Tool Message =================================
Name: get_weather

{'city': 'Mumbai', 'temp_c': 31, 'condition': 'humid, light rain'}
================================= Tool Message =================================
Name: calculator

87.8
================================== Ai Message ==================================

The current weather in Mumbai is 31 degrees Celsius with a condition of humid and light rain. 

In Fahrenheit, 31 degrees Celsius is approximately 87.8 degrees.


## 1.5 A robust tool loop

Real questions may need **several rounds** ("look up X, then compute Y from it") and tools can **fail**. A production-grade loop should:

1. Keep calling the model until it stops requesting tools.
2. Run **all** tool calls in a round (models can issue parallel calls).
3. Catch exceptions and send the error text back — the model can often recover (e.g. retry with a different argument).
4. Enforce a **maximum number of iterations** to avoid infinite loops.

In [ ]:
def run_tool_loop(llm_with_tools, messages, tools_by_name, max_iters=5, verbose=True):
    # Run the model ↔ tools loop until the model gives a final answer.
    # Mutates and returns `messages`; the last message is the final AIMessage.
    for i in range(max_iters):
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:          # model is done
            return messages

        for tc in ai_msg.tool_calls:
            tool_fn = tools_by_name.get(tc["name"])
            try:
                if tool_fn is None:
                    raise KeyError(f"Unknown tool '{tc['name']}'")
                result = tool_fn.invoke(tc["args"])
                status = "ok"
            except Exception as e:           # send the error back to the model
                result, status = f"ERROR: {e}", "error"
            if verbose:
                print(f"  [round {i+1}] {tc['name']}({tc['args']}) -> {status}: {str(result)[:80]}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"], name=tc["name"]))

    messages.append(AIMessage("Sorry, I couldn't finish that within the tool-call limit."))
    return messages

In [ ]:
# Multi-step + parallel + an error the model must recover from ("Bombay" is not in our fake DB)
msgs = [
    SystemMessage("You are a helpful assistant. Use tools when they help. If a tool fails, try to recover."),
    HumanMessage("Compare the weather in Tokyo and Bombay. Then tell me the average of the two temperatures, "
                 "and the current time in Tokyo."),
]
msgs = run_tool_loop(llm_with_tools, msgs, tools_by_name)
print("\nFINAL ANSWER:\n", text_of(msgs[-1]))

### Controlling tool choice
- `tool_choice="auto"` (default): the model decides.
- `tool_choice="any"`: must call *some* tool.
- `tool_choice="calculator"`: must call that specific tool — useful for **structured extraction**.

In [ ]:
forced = llm.bind_tools(tools, tool_choice="calculator")
print(forced.invoke("How many seconds are there in a week?").tool_calls)

### ✍️ Exercise 1
1. Write a `convert_currency(amount: float, from_ccy: str, to_ccy: str)` tool using a hard-coded rate table. Add it to the tool list and ask a question that needs *both* it and `calculator`.
2. Remove the docstring from `get_weather` and re-run 1.3. What happens to the model's tool selection? Why?
3. Ask something that could trigger an infinite loop and verify `max_iters` protects you.

In [ ]:
# Your code here

---
# Part 2 — Multi-turn Conversations

## 2.1 LLMs are stateless
Each API call is independent. The model has **no memory** of previous calls — watch:

In [ ]:
print(text_of(llm.invoke("Hi! My name is Priya and I'm learning about RAG.")))
print("-" * 60)
print(text_of(llm.invoke("What's my name?")))   # separate call → it has no idea

## 2.2 "Memory" = resending the history
The illusion of memory comes from sending the **whole conversation** each time. Message roles:

| Class | Role | Purpose |
|---|---|---|
| `SystemMessage` | system | Persona, rules, context |
| `HumanMessage` | user | What the user said |
| `AIMessage` | assistant | What the model said (may include `tool_calls`) |
| `ToolMessage` | tool | Result of a tool call |

In [ ]:
class SimpleChat:
    # Minimal multi-turn chat: keeps the full history in a list.
    def __init__(self, llm, system_prompt="You are a friendly, concise assistant."):
        self.llm = llm
        self.history = [SystemMessage(system_prompt)]

    def send(self, text: str) -> str:
        self.history.append(HumanMessage(text))
        reply = self.llm.invoke(self.history)
        self.history.append(reply)
        return text_of(reply)

chat = SimpleChat(llm)
print(chat.send("Hi! My name is Priya and I'm learning about RAG."))
print("-" * 60)
print(chat.send("What's my name, and what am I learning?"))
print(f"\nHistory length: {len(chat.history)} messages")

## 2.3 The problem: history grows forever
Every turn re-sends everything, so **cost and latency grow linearly per turn** (quadratically over the conversation) until you hit the **context window limit**. Let's measure it.

In [ ]:
def approx_tokens(messages) -> int:
    # Rough token estimate (~4 characters per token). Good enough for budgeting.
    return sum(len(text_of(m)) // 4 + 4 for m in messages)

cumulative = 0
sim = [SystemMessage("You are helpful.")]
for turn in range(1, 21):
    sim += [HumanMessage("Tell me something interesting about space. " * 3),
            AIMessage("Here is a fairly long answer about space and planets. " * 12)]
    sent = approx_tokens(sim)
    cumulative += sent
    if turn % 5 == 0:
        print(f"turn {turn:2d}: tokens sent this turn ≈ {sent:5d} | cumulative ≈ {cumulative:6d}")

## 2.4 Trimming with `trim_messages`
LangChain's `trim_messages` keeps the most recent messages that fit a token budget while preserving a valid structure (keep the system message, start on a human turn, never orphan a `ToolMessage`).

In [ ]:
from langchain_core.messages import trim_messages

trimmed = trim_messages(
    sim,
    max_tokens=400,
    token_counter=approx_tokens,   # can also pass `llm` for exact counts (provider-dependent)
    strategy="last",               # keep the most recent messages
    include_system=True,           # always keep the system prompt
    start_on="human",              # history must start with a user turn
)
print(f"Before: {len(sim)} msgs / ~{approx_tokens(sim)} tokens")
print(f"After : {len(trimmed)} msgs / ~{approx_tokens(trimmed)} tokens")
print([type(m).__name__ for m in trimmed])

Trimming is simple but **lossy**: anything older is forgotten completely. Part 3 looks at smarter strategies.

### ✍️ Exercise 2
Extend `SimpleChat` so that `send()` automatically trims history to a `max_tokens` budget before calling the model. Tell it your name, chat for ~10 turns with a small budget, then ask for your name. What happens?

In [ ]:
# Your code here

---
# Part 3 — Memory: Buffer, Summary, Vector

We'll implement every strategy behind **one small interface** so they're interchangeable:

```python
memory.add(human_text, ai_text)          # store one completed turn
memory.context(query) -> list[Message]   # what to send to the model for the next turn
```

> 💡 Older LangChain versions shipped classes like `ConversationBufferMemory` and `ConversationSummaryMemory`. These are now **deprecated** (moved to `langchain-classic`) in favour of building memory yourself or using LangGraph. Writing them by hand is also the best way to understand them.

In [ ]:
from abc import ABC, abstractmethod

class Memory(ABC):
    @abstractmethod
    def add(self, human: str, ai: str) -> None: ...

    @abstractmethod
    def context(self, query: str = "") -> list: ...

def chat_with_memory(llm, memory: Memory, user_text: str,
                     system_prompt="You are a friendly, concise assistant.") -> str:
    # One conversation turn using any Memory implementation.
    msgs = [SystemMessage(system_prompt)] + memory.context(user_text) + [HumanMessage(user_text)]
    reply = text_of(llm.invoke(msgs))
    memory.add(user_text, reply)
    return reply

# A shared script to test each memory type
SCRIPT = [
    "Hi, I'm Arjun. I'm a backend developer in Pune.",
    "My favourite language is Go, but I'm learning Python for ML.",
    "I have a dog called Pixel.",
    "What's a good first ML project for someone like me?",
    "Can you suggest a dataset for that?",
    "Thanks! Unrelated: explain what an embedding is in one sentence.",
    "Quick test — what's my dog's name and where do I work?",
]

def run_script(memory, label, script=SCRIPT, show_all=False):
    print(f"===== {label} =====")
    for i, line in enumerate(script):
        reply = chat_with_memory(llm, memory, line)
        if show_all or i == len(script) - 1:
            print(f"USER: {line}\nBOT : {reply}\n")

## 3.1 Buffer memory — keep everything
✅ Perfect recall · ❌ Unbounded cost; will eventually overflow the context window.

In [ ]:
class BufferMemory(Memory):
    def __init__(self):
        self.messages = []
    def add(self, human, ai):
        self.messages += [HumanMessage(human), AIMessage(ai)]
    def context(self, query=""):
        return list(self.messages)

buf = BufferMemory()
run_script(buf, "BufferMemory")
print("Stored messages:", len(buf.messages), "| ~tokens:", approx_tokens(buf.messages))

## 3.2 Window buffer memory — keep the last *k* turns
✅ Constant cost · ❌ Hard cutoff: older facts vanish.

In [ ]:
class WindowBufferMemory(BufferMemory):
    def __init__(self, k: int = 3):
        super().__init__()
        self.k = k
    def context(self, query=""):
        return self.messages[-2 * self.k:]   # k turns = 2k messages

run_script(WindowBufferMemory(k=2), "WindowBufferMemory (k=2)")   # expect it to forget the dog & job

## 3.3 Token buffer memory — keep as much as fits a budget
Same idea as the window, but budgeted in tokens rather than turns (more precise for variable-length messages).

In [ ]:
class TokenBufferMemory(BufferMemory):
    def __init__(self, max_tokens: int = 300):
        super().__init__()
        self.max_tokens = max_tokens
    def context(self, query=""):
        return trim_messages(self.messages, max_tokens=self.max_tokens, token_counter=approx_tokens,
                             strategy="last", start_on="human")

run_script(TokenBufferMemory(max_tokens=300), "TokenBufferMemory (300)")

## 3.4 Summary memory — compress everything into a running summary
After each turn an LLM updates a summary (a *progressive* summary: old summary + new lines → new summary).

✅ Remembers the gist of long conversations cheaply · ❌ Extra LLM call per turn, loses exact wording, summaries can drift.

In [ ]:
SUMMARY_PROMPT = '''Progressively summarize the conversation, updating the existing summary with the new lines.
Keep all concrete facts about the user (name, job, location, preferences, pets, goals) and any decisions made.
Be concise (max ~120 words). Return only the updated summary.

EXISTING SUMMARY:
{summary}

NEW LINES:
{new_lines}

UPDATED SUMMARY:'''

def format_lines(messages):
    role = {"human": "User", "ai": "Assistant"}
    return "\n".join(f"{role.get(m.type, m.type)}: {text_of(m)}" for m in messages)

class SummaryMemory(Memory):
    def __init__(self, llm):
        self.llm = llm
        self.summary = ""
    def add(self, human, ai):
        new_lines = format_lines([HumanMessage(human), AIMessage(ai)])
        prompt = SUMMARY_PROMPT.format(summary=self.summary or "(none)", new_lines=new_lines)
        self.summary = text_of(self.llm.invoke(prompt)).strip()
    def context(self, query=""):
        return [SystemMessage(f"Summary of the conversation so far:\n{self.summary}")] if self.summary else []

summ = SummaryMemory(llm)
run_script(summ, "SummaryMemory")
print("CURRENT SUMMARY:\n", summ.summary)

## 3.5 Summary-buffer memory — the hybrid (most common in practice)
Keep the **last k turns verbatim**; when a turn falls out of the window, fold it into the **summary**. You get exact recent context *and* long-term gist, and you only pay for summarization when something is evicted.

In [ ]:
class SummaryBufferMemory(Memory):
    def __init__(self, llm, k: int = 2):
        self.llm, self.k = llm, k
        self.buffer, self.summary = [], ""

    def add(self, human, ai):
        self.buffer += [HumanMessage(human), AIMessage(ai)]
        if len(self.buffer) > 2 * self.k:                    # evict oldest turns into the summary
            evicted, self.buffer = self.buffer[:-2 * self.k], self.buffer[-2 * self.k:]
            prompt = SUMMARY_PROMPT.format(summary=self.summary or "(none)", new_lines=format_lines(evicted))
            self.summary = text_of(self.llm.invoke(prompt)).strip()

    def context(self, query=""):
        ctx = [SystemMessage(f"Summary of earlier conversation:\n{self.summary}")] if self.summary else []
        return ctx + self.buffer

sb = SummaryBufferMemory(llm, k=2)
run_script(sb, "SummaryBufferMemory (k=2)")
print("SUMMARY:", sb.summary, "\nBUFFER :", len(sb.buffer), "messages")

## 3.6 Vector (retrieval) memory — recall what's *relevant*
Embed every past turn and store it in a vector store. For each new message, **retrieve the top-k most similar past turns**. This is a mini-RAG over your own conversation (next week we scale this up with ChromaDB).

✅ Scales to very long histories; recalls old but relevant facts · ❌ No sense of recency/order; retrieval can miss things phrased differently.

We use a free local embedding model (`all-MiniLM-L6-v2`, ~90 MB download on first run).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.documents import Document

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding dim:", len(embeddings.embed_query("hello")))

In [ ]:
class VectorMemory(Memory):
    def __init__(self, embeddings, k: int = 3):
        self.store = InMemoryVectorStore(embeddings)
        self.k, self.turn = k, 0

    def add(self, human, ai):
        self.turn += 1
        self.store.add_documents([Document(page_content=f"User: {human}\nAssistant: {ai}",
                                           metadata={"turn": self.turn})])

    def retrieve(self, query):
        if self.turn == 0:
            return []
        hits = self.store.similarity_search_with_score(query, k=min(self.k, self.turn))
        return sorted(hits, key=lambda h: h[0].metadata["turn"])   # present in chronological order

    def context(self, query=""):
        hits = self.retrieve(query)
        if not hits:
            return []
        recalled = "\n---\n".join(f"[turn {d.metadata['turn']}] {d.page_content}" for d, _ in hits)
        return [SystemMessage(f"Possibly relevant excerpts from earlier in this conversation:\n{recalled}")]

vec = VectorMemory(embeddings, k=2)
run_script(vec, "VectorMemory (k=2)")

print("What was retrieved for the last question:")
for doc, score in vec.retrieve("what's my dog's name and where do I work?"):
    print(f"  score={score:.3f} | turn {doc.metadata['turn']} | {doc.page_content[:70]!r}")

## 3.7 Comparison

| Memory | Recall of old facts | Recent detail | Cost / turn | Extra LLM calls | Best for |
|---|---|---|---|---|---|
| Buffer | ✅ perfect | ✅ | 📈 grows | none | Short chats, prototypes |
| Window / Token | ❌ lost after cutoff | ✅ | flat | none | Task bots, customer support |
| Summary | 🟡 gist only | 🟡 | flat | every turn | Long, flowing conversations |
| Summary-buffer | 🟡 gist | ✅ | flat | on eviction | **Sensible default** |
| Vector | ✅ if retrievable | ❌ no order | flat | none (embeddings) | Very long / multi-session memory |

In practice, strong chatbots **combine** them — which is exactly what the project does.

### ✍️ Exercise 3
1. Measure: for each memory type, print `approx_tokens(memory.context(...))` after the script. Which is cheapest? Which recalled the dog's name?
2. Make `VectorMemory` time-aware: combine similarity with a recency bonus (e.g. `score + 0.02 * turn`).

In [ ]:
# Your code here

---
# Part 4 — Conversation State Management

*Memory* is what the model sees. *State* is everything your application tracks about a conversation:

- **Session identity** — which conversation does this message belong to? (many users at once)
- **Message history** — per session
- **Structured state** — facts extracted into fields (user name, preferences, current task, slots in a booking form…)
- **Persistence** — surviving restarts

## 4.1 Multiple sessions with `RunnableWithMessageHistory`
LangChain's wrapper looks up the right history by `session_id`, injects it into the prompt, and appends the new turn automatically.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise, friendly assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

session_store: dict[str, InMemoryChatMessageHistory] = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

chat_chain = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def ask(session_id, text):
    reply = chat_chain.invoke({"input": text}, config={"configurable": {"session_id": session_id}})
    print(f"[{session_id}] USER: {text}\n[{session_id}] BOT : {text_of(reply)}\n")

ask("alice", "Hi, I'm Alice and I love hiking.")
ask("bob",   "Hey, I'm Bob. I play the guitar.")
ask("alice", "What's my hobby?")   # should say hiking — sessions are isolated
ask("bob",   "What's my name?")

print({sid: len(h.messages) for sid, h in session_store.items()})

> 📌 **Going further:** For complex, multi-step agents LangChain now recommends **LangGraph**, where state is an explicit typed object and a *checkpointer* (in-memory, SQLite, Postgres) persists it per `thread_id`. The concepts are identical to what you're learning here: session id → saved state → reload on the next message.

## 4.2 Structured state: extract facts into fields
Raw history is hard to query. We can ask the LLM to maintain a **typed user profile** using structured output (Pydantic). The profile can then be injected into the system prompt, shown in a UI, saved to a database, etc.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class UserProfile(BaseModel):
    name: Optional[str] = Field(None, description="The user's name, if they stated it")
    location: Optional[str] = Field(None, description="Where the user lives or works")
    occupation: Optional[str] = Field(None, description="The user's job or role")
    interests: list[str] = Field(default_factory=list, description="Hobbies, topics or technologies they like")
    goals: list[str] = Field(default_factory=list, description="What the user is trying to achieve")
    other_facts: list[str] = Field(default_factory=list, description="Other durable personal facts (pets, family, etc.)")

profile_extractor = llm.with_structured_output(UserProfile)

def update_profile(profile: UserProfile, human_text: str) -> UserProfile:
    # Extract new facts from one user message and merge them into the existing profile.
    new = profile_extractor.invoke(
        "Extract durable facts the USER states about themselves in the message below. "
        "Only include information explicitly stated; leave fields empty otherwise.\n\n"
        f"Message: {human_text}"
    )
    merged = profile.model_copy(deep=True)
    for field in ("name", "location", "occupation"):
        if getattr(new, field):
            setattr(merged, field, getattr(new, field))
    for field in ("interests", "goals", "other_facts"):
        current = getattr(merged, field)
        current += [x for x in getattr(new, field) if x.lower() not in {c.lower() for c in current}]
    return merged

profile = UserProfile()
for line in SCRIPT[:3]:
    profile = update_profile(profile, line)
print(profile.model_dump_json(indent=2))

## 4.3 Persistence: save & restore a session
Messages serialize cleanly with `messages_to_dict` / `messages_from_dict`. Save them together with any structured state.

In [ ]:
from pathlib import Path
from langchain_core.messages import messages_to_dict, messages_from_dict

SESSIONS_DIR = Path("chat_sessions"); SESSIONS_DIR.mkdir(exist_ok=True)

def save_session(session_id: str, history: InMemoryChatMessageHistory, profile: UserProfile | None = None):
    payload = {"session_id": session_id,
               "saved_at": datetime.now().isoformat(),
               "messages": messages_to_dict(history.messages),
               "profile": profile.model_dump() if profile else None}
    (SESSIONS_DIR / f"{session_id}.json").write_text(json.dumps(payload, indent=2))

def load_session(session_id: str):
    payload = json.loads((SESSIONS_DIR / f"{session_id}.json").read_text())
    history = InMemoryChatMessageHistory(messages=messages_from_dict(payload["messages"]))
    prof = UserProfile(**payload["profile"]) if payload["profile"] else None
    return history, prof

save_session("alice", session_store["alice"])

# Simulate a restart: wipe memory, then restore from disk
session_store.clear()
session_store["alice"], _ = load_session("alice")
ask("alice", "Remind me — what did I tell you I love?")

### ✍️ Exercise 4
Add a `current_task` field to the state (e.g. `"planning a trip"`, `"debugging code"`) and have the bot update it each turn. How would you use this to change the system prompt dynamically?

In [ ]:
# Your code here

---
# 🎯 Project — A 1-class Chatbot with Full Memory Handling

Everything from this week in **one class**, `MemoryChatbot`:

| Capability | How |
|---|---|
| **Tool calling** | `bind_tools` + the robust loop from Part 1 |
| **Short-term memory** | Window buffer of the last *k* turns (verbatim) |
| **Mid-term memory** | Running summary of turns evicted from the window |
| **Long-term memory** | Vector store of every turn, retrieved by relevance |
| **Structured state** | A `UserProfile` updated from each user message |
| **Session management** | `session_id`, `save()` / `load()` to JSON, `reset()` |
| **Observability** | `stats()` and optional debug printing of the assembled context |

### How each turn is assembled

```
SystemMessage ── persona + tool rules
              ── user profile (structured state)
              ── running summary (mid-term)
              ── recalled excerpts (long-term, vector search on the new input)
+ last k turns (short-term buffer)
+ HumanMessage(new input)
→ tool loop → final answer
→ update buffer, evict→summary, embed turn, update profile, autosave
```

In [ ]:
class MemoryChatbot:
    """A chatbot with tools, layered memory (buffer + summary + vector), structured state and persistence."""

    BASE_PROMPT = ("You are a helpful, friendly assistant. Use tools when they help and never invent tool results. "
                   "Use the memory sections below to personalize answers, but don't recite them unprompted.")

    def __init__(self, llm, tools=None, embeddings=None, session_id="default",
                 window_turns=4, recall_k=3, extract_profile=True,
                 storage_dir="chat_sessions", autosave=True, max_tool_iters=5, debug=False):
        self.llm = llm
        self.tools = list(tools or [])
        self.tools_by_name = {t.name: t for t in self.tools}
        self.llm_with_tools = llm.bind_tools(self.tools) if self.tools else llm
        self.embeddings = embeddings
        self.session_id = session_id
        self.window_turns, self.recall_k = window_turns, recall_k
        self.extract_profile = extract_profile
        self.storage_dir = Path(storage_dir); self.storage_dir.mkdir(parents=True, exist_ok=True)
        self.autosave, self.max_tool_iters, self.debug = autosave, max_tool_iters, debug
        self.reset()

    # ------------------------------------------------------------------ public API
    def chat(self, user_input: str) -> str:
        """Run one full conversation turn and return the assistant's reply."""
        self.turn += 1
        recalled = self._recall(user_input)
        messages = [SystemMessage(self._system_prompt(recalled)), *self.buffer, HumanMessage(user_input)]
        if self.debug:
            print(f"--- context: {len(messages)} msgs, ~{approx_tokens(messages)} tokens, "
                  f"{len(recalled)} recalled ---")

        reply = self._run_tools(messages)

        # Only store the clean human/AI pair (not intermediate tool messages) in memory
        self.buffer += [HumanMessage(user_input), AIMessage(reply)]
        self._remember(user_input, reply)
        self._evict_to_summary()
        if self.extract_profile:
            self._update_profile(user_input)
        if self.autosave:
            self.save()
        return reply

    def reset(self):
        """Clear all memory and state for this session (does not delete saved files)."""
        self.buffer: list[BaseMessage] = []
        self.summary = ""
        self.profile = UserProfile()
        self.long_term: list[dict] = []           # raw texts, so the vector store can be rebuilt on load
        self.store = InMemoryVectorStore(self.embeddings) if self.embeddings else None
        self.turn = 0
        self.tool_log: list[dict] = []

    def stats(self) -> dict:
        return {"session_id": self.session_id, "turns": self.turn,
                "buffer_messages": len(self.buffer), "summary_chars": len(self.summary),
                "long_term_memories": len(self.long_term), "tool_calls": len(self.tool_log),
                "profile": self.profile.model_dump(exclude_defaults=True)}

    # ------------------------------------------------------------------ persistence
    @property
    def path(self) -> Path:
        return self.storage_dir / f"{self.session_id}.json"

    def save(self):
        payload = {"session_id": self.session_id, "turn": self.turn, "saved_at": datetime.now().isoformat(),
                   "buffer": messages_to_dict(self.buffer), "summary": self.summary,
                   "profile": self.profile.model_dump(), "long_term": self.long_term,
                   "tool_log": self.tool_log}
        self.path.write_text(json.dumps(payload, indent=2, default=str))

    @classmethod
    def load(cls, session_id: str, llm, **kwargs) -> "MemoryChatbot":
        bot = cls(llm, session_id=session_id, **kwargs)
        if not bot.path.exists():
            print(f"(no saved session '{session_id}', starting fresh)")
            return bot
        data = json.loads(bot.path.read_text())
        bot.turn, bot.summary = data["turn"], data["summary"]
        bot.buffer = messages_from_dict(data["buffer"])
        bot.profile = UserProfile(**data["profile"])
        bot.tool_log = data.get("tool_log", [])
        bot.long_term = data["long_term"]
        if bot.store is not None and bot.long_term:          # rebuild the vector index
            bot.store.add_documents([Document(page_content=m["text"], metadata={"turn": m["turn"]})
                                     for m in bot.long_term])
        return bot

    # ------------------------------------------------------------------ internals
    def _system_prompt(self, recalled: list[str]) -> str:
        parts = [self.BASE_PROMPT]
        prof = self.profile.model_dump(exclude_defaults=True)
        if prof:
            parts.append("## What you know about the user\n" + json.dumps(prof, indent=1))
        if self.summary:
            parts.append("## Summary of earlier conversation\n" + self.summary)
        if recalled:
            parts.append("## Possibly relevant excerpts from past turns\n" + "\n---\n".join(recalled))
        parts.append(f"(Current time UTC: {datetime.now(ZoneInfo('UTC')):%Y-%m-%d %H:%M})")
        return "\n\n".join(parts)

    def _run_tools(self, messages) -> str:
        for _ in range(self.max_tool_iters):
            ai_msg = self.llm_with_tools.invoke(messages)
            messages.append(ai_msg)
            if not getattr(ai_msg, "tool_calls", None):
                return text_of(ai_msg)
            for tc in ai_msg.tool_calls:
                tool_fn = self.tools_by_name.get(tc["name"])
                try:
                    if tool_fn is None:
                        raise KeyError(f"Unknown tool '{tc['name']}'")
                    result, ok = tool_fn.invoke(tc["args"]), True
                except Exception as e:
                    result, ok = f"ERROR: {e}", False
                self.tool_log.append({"turn": self.turn, "tool": tc["name"], "args": tc["args"], "ok": ok})
                if self.debug:
                    print(f"  🔧 {tc['name']}({tc['args']}) -> {str(result)[:80]}")
                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"], name=tc["name"]))
        return "Sorry — I couldn't complete that within the tool-call limit."

    def _recall(self, query: str) -> list[str]:
        """Long-term memory: vector search over turns that are NOT already in the buffer."""
        if self.store is None or not self.long_term:
            return []
        cutoff = self.turn - self.window_turns     # turns >= cutoff are still verbatim in the buffer
        hits = self.store.similarity_search(query, k=self.recall_k + self.window_turns)
        hits = [d for d in hits if d.metadata["turn"] < cutoff][: self.recall_k]
        hits.sort(key=lambda d: d.metadata["turn"])
        return [f"[turn {d.metadata['turn']}] {d.page_content}" for d in hits]

    def _remember(self, human: str, ai: str):
        text = f"User: {human}\nAssistant: {ai}"
        self.long_term.append({"turn": self.turn, "text": text})
        if self.store is not None:
            self.store.add_documents([Document(page_content=text, metadata={"turn": self.turn})])

    def _evict_to_summary(self):
        """Mid-term memory: fold turns that fall out of the window into the running summary."""
        max_msgs = 2 * self.window_turns
        if len(self.buffer) <= max_msgs:
            return
        evicted, self.buffer = self.buffer[:-max_msgs], self.buffer[-max_msgs:]
        prompt = SUMMARY_PROMPT.format(summary=self.summary or "(none)", new_lines=format_lines(evicted))
        self.summary = text_of(self.llm.invoke(prompt)).strip()
        if self.debug:
            print(f"  📝 summarized {len(evicted)} old messages")

    def _update_profile(self, human: str):
        try:
            self.profile = update_profile(self.profile, human)
        except Exception as e:           # structured output can fail on small local models — don't crash the chat
            if self.debug:
                print(f"  ⚠️ profile update skipped: {e}")

## Demo 1 — A conversation that stresses every memory layer
We use a **small window (2 turns)** on purpose, so early facts are quickly evicted and must be recovered from the summary, the vector store or the profile.

In [ ]:
bot = MemoryChatbot(llm, tools=tools, embeddings=embeddings, session_id="demo_user",
                    window_turns=2, recall_k=2, debug=True)

conversation = [
    "Hi! I'm Meera, a data analyst based in Bengaluru. I have a cat named Biscuit.",
    "I'm preparing for ML interviews next month — mostly focused on NLP roles.",
    "What's the weather in Delhi? I'm flying there for an interview.",
    "If my flight is 2h 35m and I have a 1h 50m layover, how many minutes is that in total?",
    "Give me one tip for explaining transformers in an interview.",
    "What time is it in London right now? I have a call with a recruiter there.",
    "OK, memory check: what's my cat's name, which city do I live in, and what roles am I targeting?",
]

for msg in conversation:
    print(f"\n👤 {msg}")
    print(f"🤖 {bot.chat(msg)}")

In [ ]:
# Inspect every memory layer
print("STATS:", json.dumps(bot.stats(), indent=2), "\n")
print("SUMMARY (mid-term):\n", bot.summary, "\n")
print("BUFFER (short-term):"); show(bot.buffer)

## Demo 2 — Persistence across a "restart"

In [ ]:
del bot   # simulate the program shutting down

restored = MemoryChatbot.load("demo_user", llm, tools=tools, embeddings=embeddings,
                              window_turns=2, recall_k=2, debug=True)
print("Restored:", restored.stats()["turns"], "turns\n")
print("🤖", restored.chat("I'm back! Remind me — what was I preparing for, and where was I flying?"))

## Demo 3 — Interactive chat loop
Commands: `/stats`, `/summary`, `/reset`, `/save`, `/quit`.

In [ ]:
def chat_loop(bot: MemoryChatbot):
    print(f"Chatting in session '{bot.session_id}'. Type /quit to exit.")
    while True:
        user = input("👤 You: ").strip()
        if not user:
            continue
        if user == "/quit":
            bot.save(); print("Saved. Bye!"); break
        elif user == "/stats":
            print(json.dumps(bot.stats(), indent=2))
        elif user == "/summary":
            print(bot.summary or "(no summary yet)")
        elif user == "/reset":
            bot.reset(); print("Memory cleared.")
        elif user == "/save":
            bot.save(); print(f"Saved to {bot.path}")
        else:
            print("🤖 Bot:", bot.chat(user))

# Uncomment to try it:
# chat_loop(MemoryChatbot.load("my_session", llm, tools=tools, embeddings=embeddings))

## Project extensions (pick at least two)

1. **Memory tool** — give the bot a `save_note(note: str)` tool so it can *decide* to store important facts in long-term memory, and a `search_notes(query: str)` tool to look them up on demand (this is how "agentic memory" works).
2. **Token budget** — replace the fixed `window_turns` with a token budget using `trim_messages`, and log the token count of every request.
3. **Cross-session profile** — store the `UserProfile` per *user* rather than per *session*, so a new conversation still knows who the user is.
4. **Forgetting** — add a `/forget <fact>` command that removes matching entries from the profile and long-term memory (think: privacy & GDPR).
5. **Evaluation** — write 10 "memory check" questions with expected answers and score how often the bot recalls correctly with each memory layer switched off (ablation study). Reuse your evaluator from Week 10!
6. **UI** — wrap the class in a Gradio or Streamlit chat interface with a side panel showing the profile, summary and recalled memories live.
7. **ChromaDB** — swap `InMemoryVectorStore` for a persistent Chroma collection (a warm-up for Week 12).

In [ ]:
# Your extensions here

---
# ✅ Week 11 Recap

- **Tool calling** is a protocol: the model emits structured requests (`tool_calls`), *your* code executes them, results return as `ToolMessage`s. Loop until no more calls; handle errors; cap iterations.
- **LLMs are stateless** — multi-turn chat works by resending history, which grows in cost and eventually overflows the context window.
- **Memory strategies** trade recall vs. cost: buffer → window/token → summary → summary-buffer → vector. Real systems **layer** them.
- **State management** means isolating sessions, extracting structured facts, and persisting everything so conversations survive restarts.

### Self-check questions
1. Why must a `ToolMessage` carry a `tool_call_id`?
2. What happens if you send a `ToolMessage` without the preceding `AIMessage` that requested it?
3. When would summary memory perform *worse* than a window buffer?
4. Why does `MemoryChatbot._recall` skip turns that are still in the buffer?
5. What privacy concerns arise from long-term vector memory, and how would you address them?

**Next week → Week 12: RAG Pipeline** — the vector memory you built here becomes a full retrieval system over PDF documents with ChromaDB, chunking and reranking.